### Batch Data Ingestion with CTAS

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS dbacademy
COMMENT 'catalog for databrikcs academy courses';

CREATE SCHEMA IF NOT EXISTS dbacademy.labs
COMMENT 'schema for databricks academy courses labs';

USE CATALOG dbacademy;
USE SCHEMA labs;

In [0]:
%sql
SELECT current_catalog(), current_schema()

In [0]:
%sql
SELECT *
FROM parquet.`/Volumes/dbacademy_ecommerce/v01/raw/users-historical`;

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/dbacademy_ecommerce/v01/raw/users-historical',
  format => 'parquet'
)
LIMIT 10;

In [0]:
%sql
-- Drop the table if it exists
DROP TABLE IF EXISTS historical_users_bronze_ctas_rf;

-- Create the Delta table
CREATE TABLE historical_users_bronze_ctas_rf AS
SELECT *
FROM read_files(
  '/Volumes/dbacademy_ecommerce/v01/raw/users-historical',
  format => 'parquet'
);

-- Preview the Delta table
SELECT *
FROM historical_users_bronze_ctas_rf
LIMIT 10;

In [0]:
%sql
DESCRIBE EXTENDED historical_users_bronze_ctas_rf;

### Bonus - Doing the same thing in python

In [0]:
%python
# 1. Read the parquet files from the volume into Spark DataFrame
df = (spark
      .read
      .format("parquet")
      .load("/Volumes/dbacademy_ecommerce/v01/raw/users-historical")
    )

# 2. Write the DataFrame to Delta table (overwrite the table if it exists)
(df
 .write
 .mode("overwrite")
 .saveAsTable(f"dbacademy.labs.historical_users_bronze_python")
)

# 3. Read and view the table
users_bronze_table = spark.table(f"dbacademy.labs.historical_users_bronze_python")
display(users_bronze_table)

### Incremental Data Ingestion with COPY INTO

In [0]:
----------------------------------
-- This cell returns an error: due to schema evolution
----------------------------------

-- Drop the table if it exists for demonstrations purposes
DROP TABLE IF EXISTS historical_users_bronze_ci;

-- Create an empty table with the specified table schema (ony 2 out of 3 columns)
CREATE TABLE historical_users_bronze_ci (
  user_id STRING,
  user_first_touch_timestamp BIGINT
);

-- Use COPY INTO to populate Delata table
COPY INTO historical_users_bronze_ci
  FROM '/Volumes/dbacademy_ecommerce/v01/raw/users-historical'
  FILEFORMAT = parquet;

In [0]:
COPY INTO historical_users_bronze_ci
  FROM '/Volumes/dbacademy_ecommerce/v01/raw/users-historical'
  FILEFORMAT = parquet;
  COPY_OPTIONS ('mergeSchema' = 'true' ); -- Merge the schema of each file 

In [0]:
SELECT *
FROM historical_users_bronze_ci
LIMIT 10;

In [0]:
-- Drop the table if it exists for demonstrations purposes
DROP TABLE IF EXISTS historical_users_bronze_ci_no_schema;

-- Create an empty table without the specified schema
CREATE TABLE historical_users_bronze_ci_no_schema;

-- Use COPY INTO to populate Delata table
COPY INTO historical_users_bronze_ci_no_schema
  FROM '/Volumes/dbacademy_ecommerce/v01/raw/users-historical'
  FILEFORMAT = parquet;
  COPY_OPTIONS ('mergeSchema' = 'true' );